# 07. Divide and Conquer and the Master Theorem

**Divide and conquer:** split the problem into **independent** subproblems, solve each recursively, **combine** the results. Examples: merge sort, quicksort, binary search, fast power, Karatsuba multiplication.

**Learning order:** PATTERN → MASTER THEOREM → CALCULATOR → MAX SUBARRAY → MAJORITY → KARATSUBA → D&C VS DP → INTERVIEW THINKING

## 1. The pattern

1. **Divide** into `a` subproblems of size `n / b`.
2. **Conquer** each recursively (base case for tiny inputs).
3. **Combine** in `f(n)` time.

Recurrence: `T(n) = a · T(n / b) + f(n)`.

**Difference from DP:** D&C subproblems are **independent** (no overlap). If they overlap, memoize and call it DP.

## 2. The Master Theorem (for `f(n) = n^k`)

Compare the **recursion's leaf work** `n^c` where `c = log_b(a)` with the **combine work** `n^k`:

| Case | Who dominates | Result |
|---|---|---|
| `k < c` | leaves | `Θ(n^c)` |
| `k = c` | all levels equal | `Θ(n^k log n)` |
| `k > c` | the root | `Θ(n^k)` |

| Algorithm | `a, b, k` | Result |
|---|---|---|
| binary search | 1, 2, 0 | `Θ(log n)` |
| merge sort | 2, 2, 1 | `Θ(n log n)` |
| max by splitting in halves | 2, 2, 0 | `Θ(n)` |
| Karatsuba | 3, 2, 1 | `Θ(n^1.585)` |
| Strassen | 7, 2, 2 | `Θ(n^2.807)` |
| naive 8-way matrix multiply | 8, 2, 2 | `Θ(n³)` |

In [1]:
import math

def master_theorem(a, b, k):
    c = math.log(a, b)
    if math.isclose(c, k):
        return f"Theta(n^{k} log n)" if k else "Theta(log n)"
    if k < c:
        return f"Theta(n^{round(c, 3)})"
    return f"Theta(n^{k})"

print(master_theorem(1, 2, 0), master_theorem(2, 2, 1), master_theorem(2, 2, 0))   # Theta(log n) Theta(n^1 log n) Theta(n^1.0)
print(master_theorem(3, 2, 1), master_theorem(8, 2, 2), master_theorem(1, 2, 1))   # Theta(n^1.585) Theta(n^3.0) Theta(n^1)

Theta(log n) Theta(n^1 log n) Theta(n^1.0)
Theta(n^1.585) Theta(n^3.0) Theta(n^1)


## 3. Maximum Subarray by divide and conquer

The best subarray is entirely in the **left half**, entirely in the **right half**, or **crosses the middle**. The crossing best = (best suffix sum of the left half) + (best prefix sum of the right half). `T(n) = 2T(n/2) + O(n)` → O(n log n). (Kadane is O(n); this shows the technique.)

In [2]:
def max_subarray_dc(nums, lo=0, hi=None):
    if hi is None:
        hi = len(nums) - 1
    if lo == hi:
        return nums[lo]
    mid = (lo + hi) // 2
    left_best = max_subarray_dc(nums, lo, mid)
    right_best = max_subarray_dc(nums, mid + 1, hi)
    s, left_cross = 0, float("-inf")
    for i in range(mid, lo - 1, -1):                          # best suffix of the left half
        s += nums[i]
        left_cross = max(left_cross, s)
    s, right_cross = 0, float("-inf")
    for i in range(mid + 1, hi + 1):                          # best prefix of the right half
        s += nums[i]
        right_cross = max(right_cross, s)
    return max(left_best, right_best, left_cross + right_cross)

print(max_subarray_dc([-2, 1, -3, 4, -1, 2, 1, -5, 4]), max_subarray_dc([-3, -1, -2]))   # 6 -1

6 -1


## 4. Majority Element by divide and conquer

The majority of the whole array must be the majority of **at least one half**. Get the candidate of each half; if they differ, **count** both in the whole range and keep the one with more. `T(n) = 2T(n/2) + O(n)`.

In [3]:
def majority_dc(nums, lo=0, hi=None):
    if hi is None:
        hi = len(nums) - 1
    if lo == hi:
        return nums[lo]
    mid = (lo + hi) // 2
    left, right = majority_dc(nums, lo, mid), majority_dc(nums, mid + 1, hi)
    if left == right:
        return left
    count_left = sum(1 for i in range(lo, hi + 1) if nums[i] == left)
    count_right = sum(1 for i in range(lo, hi + 1) if nums[i] == right)
    return left if count_left > count_right else right

print(majority_dc([2, 2, 1, 1, 1, 2, 2]), majority_dc([3, 2, 3]))   # 2 3

2 3


## 5. Karatsuba multiplication (3 multiplications instead of 4)

Split `x = a·10^m + b`, `y = c·10^m + d`. The school method needs `ac`, `ad`, `bc`, `bd`. Karatsuba needs only `ac`, `bd` and `(a + b)(c + d)`, because `ad + bc = (a + b)(c + d) - ac - bd`. So `T(n) = 3T(n/2) + O(n)` = `Θ(n^1.585)`.

In [4]:
def karatsuba(x, y):
    if x < 10 or y < 10:
        return x * y
    m = max(len(str(x)), len(str(y))) // 2
    a, b = divmod(x, 10 ** m)
    c, d = divmod(y, 10 ** m)
    ac = karatsuba(a, c)
    bd = karatsuba(b, d)
    cross = karatsuba(a + b, c + d) - ac - bd              # = ad + bc
    return ac * 10 ** (2 * m) + cross * 10 ** m + bd

x, y = 12345678901234567890, 98765432109876543210
print(karatsuba(1234, 5678), karatsuba(x, y) == x * y)   # 7006652 True

7006652 True


## 6. Divide and conquer vs DP vs greedy

| | D&C | DP | Greedy |
|---|---|---|---|
| subproblems | independent | overlapping | one choice |
| stores results | no | yes | no |
| typical | sorting, searching, closest pair, FFT | LCS, knapsack | scheduling, MST |

Other D&C classics: **quickselect** (average O(n)), **count inversions** (merge sort), **closest pair of points** (O(n log n)), **FFT**, **Strassen**, **skyline problem**, **median of medians** (worst-case O(n) selection).

## 7. Interview thinking

**Say:** "I split the array in half, solve both halves, and combine in O(n). So `T(n) = 2T(n/2) + O(n)`, which is O(n log n) by the Master Theorem. Recursion depth is log n."

**Common mistakes:** combining in more than the claimed time (hidden slicing copies), unbalanced splits (worst-case O(n²) like quicksort on sorted data), forgetting the base case for 0/1 elements, applying D&C when subproblems overlap (use DP).

## Quick Revision

- `T(n) = a·T(n/b) + n^k`: compare `k` with `log_b a`. `<` leaves, `=` extra log, `>` root.
- Merge sort `Θ(n log n)`, binary search `Θ(log n)`, Karatsuba `Θ(n^1.585)`.
- Max subarray / majority: left, right, **crossing** case.
- Independent subproblems → D&C. Overlapping → DP.